In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

In [2]:
import sys; print(sys.executable)

c:\Users\realj\AppData\Local\Programs\Python\Python314\python.exe


In [ ]:
SPLIT = "train" # "train" or "test"

In [28]:
train_base_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_base.csv')
train_static_0_0_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_static_0_0.csv')
train_static_0_1_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_static_0_1.csv')
train_static_cb_0_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_static_cb_0.csv')
train_person_1_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_person_1.csv')
train_credit_bureau_a_1_0_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_credit_bureau_a_1_0.csv')
train_credit_bureau_a_1_1_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_credit_bureau_a_1_1.csv')
train_credit_bureau_a_2_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_credit_bureau_a_2.csv')
train_applprev_1_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_applprev_1.csv')
train_tax_registry_a_1_df = pd.read_csv(f'../data/csv_files/{SPLIT}/{SPLIT}_tax_registry_a_1.csv')

In [29]:
# Concatenate the two credit bureau dataframes into one
train_credit_bureau_a_1_df = pd.concat([train_credit_bureau_a_1_0_df, train_credit_bureau_a_1_1_df], 
                                       axis=0, ignore_index=True)
# Concatenate the two static dataframes into one
train_static_0_df = pd.concat([train_static_0_0_df, train_static_0_1_df], 
                                       axis=0, ignore_index=True)

In [30]:
# Sort so num_group1 == 0 comes first for every case_id
train_person_1_df = train_person_1_df.sort_values(by=["case_id", "num_group1"])

# Aggregate person-level data to case-level data
person_agg = train_person_1_df.groupby("case_id").agg(
    num_applicants=("num_group1", "count"),
    max_income=("mainoccupationinc_384A", "max"),
    sum_income=("mainoccupationinc_384A", "sum"),
    primary_housing_type=("housingtype_772M", "first")
).reset_index()

# Aggregate credit bureau features per applicant
cb1_agg = train_credit_bureau_a_1_df.groupby("case_id").agg(
    # Thin-file signal (0 = Thin file, 1+ = Established)
    bureau_rowcount=("num_group1", "count"),
    
    # Overdue Amount: Both peak and average
    max_overdue_amount=("overdueamountmax_950A", "max"),
    mean_overdue_amount=("overdueamountmax_950A", "mean"),
    
    # Days Past Due: Peak severity and average discipline
    max_days_past_due=("pmts_dpdvalue_108P", "max"),
    mean_days_past_due=("pmts_dpdvalue_108P", "mean"),
    
    # Credit Amounts: Average capacity and peak exposure
    mean_past_credit_limit=("credamount_770A", "mean"),
    max_past_credit_limit=("credamount_770A", "max")
).reset_index()

# Percentage of previous contracts with any days past due
dpd_rate = (
    train_credit_bureau_a_1_df
    .assign(has_dpd=lambda x: (x["pmts_dpdvalue_108P"] > 0).astype(int))
    .groupby("case_id")["has_dpd"]
    .mean()
    .reset_index(name="dpd_contract_rate")
)

# Percentage of previous contracts with any overdue amount
overdue_rate = (
    train_credit_bureau_a_1_df
    .assign(has_overdue=lambda x: (x["overdueamountmax_950A"] > 0).astype(int))
    .groupby("case_id")["has_overdue"]
    .mean()
    .reset_index(name="overdue_contract_rate")
)

# Add the two new features to the bureau summary
cb1_agg = cb1_agg.merge(dpd_rate, on="case_id", how="left")
cb1_agg = cb1_agg.merge(overdue_rate, on="case_id", how="left")


# Aggregate monthly bureau history to the contract level first
cb2_contract_agg = train_credit_bureau_a_2_df.groupby(
    ["case_id", "num_group1"]
).agg(
    monthly_record_count=("pmts_month_158T", "count"),
    max_monthly_overdue=("pmts_overdue_1140A", "max"),
    mean_monthly_overdue=("pmts_overdue_1140A", "mean"),
    sum_monthly_overdue=("pmts_overdue_1140A", "sum")
).reset_index()

# Then aggregate the contract-level information to the applicant level
cb2_agg = cb2_contract_agg.groupby("case_id").agg(
    total_monthly_records=("monthly_record_count", "sum"),
    max_monthly_overdue=("max_monthly_overdue", "max"),
    mean_monthly_overdue=("mean_monthly_overdue", "mean"),
    sum_monthly_overdue=("sum_monthly_overdue", "sum"),
    num_bureau_contracts_with_monthly_history=("num_group1", "count")
).reset_index()

# Average number of monthly records per credit contract
cb2_agg["months_per_contract"] = (
    cb2_agg["total_monthly_records"] /
    cb2_agg["num_bureau_contracts_with_monthly_history"]
)

train_applprev_1_df["approvaldate_319D"] = pd.to_datetime(train_applprev_1_df["approvaldate_319D"]
                                                          , errors="coerce")

# Aggregate prior application features
applprev_agg = train_applprev_1_df.groupby("case_id").agg(
    
    prior_app_count=("num_group1", "count"),
    
    # Requested Amount signals (peak, average, and most recent)
    max_prior_requested_amt=("credamount_590A", "max"),
    mean_prior_requested_amt=("credamount_590A", "mean"),
    
    # Recency Signal
    most_recent_approval_date=("approvaldate_319D", "max"),
).reset_index()

# Aggregate tax registry features per applicant
tax_agg = train_tax_registry_a_1_df.groupby("case_id").agg(
    tax_filing_count=("num_group1", "count"),
    mean_tax_income=("amount_4527230A", "mean"),
    max_tax_income=("amount_4527230A", "max"),
    num_unique_employers=("name_4527232M", "nunique")
).reset_index()

In [31]:
# Create a master dataframe to hold all features
df_master = train_base_df.copy()

# Merge all aggregated features into the master dataframe
df_master = df_master.merge(train_static_0_df, on="case_id", how="left")
df_master = df_master.merge(train_static_cb_0_df, on="case_id", how="left")
df_master = df_master.merge(person_agg, on="case_id", how="left")
df_master = df_master.merge(cb1_agg, on="case_id", how="left")
df_master = df_master.merge(cb2_agg, on="case_id", how="left")
df_master = df_master.merge(applprev_agg, on="case_id", how="left")
df_master = df_master.merge(tax_agg, on="case_id", how="left")

In [32]:
# Fill NaN values for numeric columns with 0 to extract thin-file segment flag
df_master["bureau_rowcount"] = df_master["bureau_rowcount"].fillna(0)
df_master["total_monthly_records"] = df_master["total_monthly_records"].fillna(0)
df_master["prior_app_count"] = df_master["prior_app_count"].fillna(0)
df_master["tax_filing_count"] = df_master["tax_filing_count"].fillna(0)
# Thin-File Segment Flag
df_master["is_thin_file"] = (df_master["bureau_rowcount"] == 0).astype(int)

In [33]:
df_master.head(10)

,case_id,date_decision,MONTH,WEEK_NUM,mainoccupationinc_384A,credamount_770A,annuity_780A,days_employed_700P,education_927M,maritalstatus_703M,...,months_per_contract,prior_app_count,max_prior_requested_amt,mean_prior_requested_amt,most_recent_approval_date,tax_filing_count,mean_tax_income,max_tax_income,num_unique_employers,is_thin_file
0,1000001,2020-11-03,202011,96,28384.0,7397.0,1188.0,661.0,2d4a2bc4,ea6fb4b5,...,NaN,4.0,13384.0,7152.250000,2018-01-23,1.0,29089.0,29089.0,1.0,1
1,1000002,2020-10-27,202010,95,40770.0,16124.0,637.0,2766.0,f937ecaa,b58630a7,...,1.857143,3.0,6917.0,4888.333333,2018-04-21,0.0,NaN,NaN,NaN,0
2,1000003,2020-10-06,202010,92,25052.0,30771.0,749.0,0.0,6def22f0,b58630a7,...,2.000000,0.0,NaN,NaN,NaT,3.0,18017.0,21291.0,3.0,0
3,1000004,2020-12-15,202012,102,16929.0,11339.0,1467.0,1768.0,242b264b,ea6fb4b5,...,2.000000,3.0,10327.0,8464.333333,2018-11-10,0.0,NaN,NaN,NaN,0
4,1000005,2020-11-17,202011,98,20529.0,15124.0,1760.0,2067.0,f937ecaa,e78cb9c0,...,NaN,2.0,12076.0,9346.500000,2017-02-24,2.0,18889.0,24569.0,2.0,1
5,1000006,2020-12-22,202012,103,17856.0,11933.0,382.0,3586.0,242b264b,b58630a7,...,NaN,3.0,8857.0,7879.333333,2017-10-13,1.0,19395.0,19395.0,1.0,1
6,1000007,2020-12-22,202012,103,62811.0,NaN,1592.0,2178.0,10795bac,e78cb9c0,...,3.000000,1.0,12570.0,12570.000000,2017-02-11,1.0,34844.0,34844.0,1.0,0
7,1000008,2020-12-29,202012,104,24109.0,8627.0,1686.0,2643.0,10795bac,ea6fb4b5,...,NaN,4.0,21920.0,10075.500000,2018-06-23,1.0,47214.0,47214.0,1.0,1
8,1000009,2020-12-08,202012,101,38064.0,16009.0,1027.0,1049.0,f937ecaa,ea6fb4b5,...,NaN,2.0,11710.0,9610.500000,2018-07-15,3.0,36070.0,50469.0,3.0,1
9,1000010,2020-11-17,202011,98,18313.0,54673.0,852.0,1881.0,2d4a2bc4,ea6fb4b5,...,NaN,0.0,NaN,NaN,NaT,2.0,23819.0,30187.0,2.0,1


In [34]:
# Missing value count and percentage for every column in df_master
missing_df = pd.DataFrame({
    "missing_count": df_master.isnull().sum(),
    "pct_missing": (df_master.isnull().mean() * 100).round(2)
})

print(missing_df)

                                           missing_count  pct_missing
case_id                                                0         0.00
date_decision                                          0         0.00
MONTH                                                  0         0.00
WEEK_NUM                                               0         0.00
mainoccupationinc_384A                             10021         5.01
credamount_770A                                     3952         1.98
annuity_780A                                       20086        10.04
days_employed_700P                                 30030        15.02
education_927M                                         0         0.00
maritalstatus_703M                                     0         0.00
birth_259D                                             0         0.00
riskassesment_940T                                 66954        33.48
numberofqueries_146L                               60029        30.01
description_5085714M

In [35]:
# percentage of thin-file applicants
df_master["is_thin_file"].mean() * 100

np.float64(30.014499999999998)

In [36]:
missing_df = pd.DataFrame({
    "missing_count": df_master.isnull().sum(),
    "pct_missing": (df_master.isnull().mean() * 100).round(2)
})

print(missing_df)


                                           missing_count  pct_missing
case_id                                                0         0.00
date_decision                                          0         0.00
MONTH                                                  0         0.00
WEEK_NUM                                               0         0.00
mainoccupationinc_384A                             10021         5.01
credamount_770A                                     3952         1.98
annuity_780A                                       20086        10.04
days_employed_700P                                 30030        15.02
education_927M                                         0         0.00
maritalstatus_703M                                     0         0.00
birth_259D                                             0         0.00
riskassesment_940T                                 66954        33.48
numberofqueries_146L                               60029        30.01
description_5085714M

In [37]:
# Some checks

print(df_master.shape)
print(missing_df.shape)
print(missing_df.head())

print(df_master.isna().sum().sum())
if SPLIT == "train": print(df_master["target"].mean())


has_null = df_master.isna().any(axis=1)
print(pd.crosstab(df_master["is_thin_file"], has_null))

if SPLIT == "train": print(df_master.groupby("is_thin_file")["target"].agg(["mean", "count"]))

(200000, 42)
(42, 2)
                        missing_count  pct_missing
case_id                             0         0.00
date_decision                       0         0.00
MONTH                               0         0.00
WEEK_NUM                            0         0.00
mainoccupationinc_384A          10021         5.01
1367372
col_0         False  True 
is_thin_file              
0             47328  92643
1                 0  60029


In [38]:
print(df_master.columns.tolist())

['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'mainoccupationinc_384A', 'credamount_770A', 'annuity_780A', 'days_employed_700P', 'education_927M', 'maritalstatus_703M', 'birth_259D', 'riskassesment_940T', 'numberofqueries_146L', 'description_5085714M', 'num_applicants', 'max_income', 'sum_income', 'primary_housing_type', 'bureau_rowcount', 'max_overdue_amount', 'mean_overdue_amount', 'max_days_past_due', 'mean_days_past_due', 'mean_past_credit_limit', 'max_past_credit_limit', 'dpd_contract_rate', 'overdue_contract_rate', 'total_monthly_records', 'max_monthly_overdue', 'mean_monthly_overdue', 'sum_monthly_overdue', 'num_bureau_contracts_with_monthly_history', 'months_per_contract', 'prior_app_count', 'max_prior_requested_amt', 'mean_prior_requested_amt', 'most_recent_approval_date', 'tax_filing_count', 'mean_tax_income', 'max_tax_income', 'num_unique_employers', 'is_thin_file']


# Feature Engineering

See [Model_Feature_Catalog (1).pdf](https://drive.google.com/file/d/1dNC1biNsUu6kase-q9nfG5P9fn0vEFjo/view?usp=sharing)

In [39]:
'''
Family 1 · Raw applicant facts 4 features
The headline facts from static_0 — income, requested credit, annuity, employment length — joined directly. These are the raw
inputs and the denominators most other families build on.
'''

# Restore original source values here because earlier cells imputed some numeric fields.
# Ratios and missing flags below must use the values before that imputation.
'''
days_employed_700P
Length of employment, in days.

Job stability proxy; longer tenure → lower risk. Direct join from static_0.
'''

'''
mainoccupationinc_384A
Stated applicant income.

Core repayment capacity and the denominator of
most ratios; higher income → lower default.

Direct join from static_0 on
case_id.
'''

'''
annuity_780A
The loan's periodic payment.

The actual cash-flow obligation; only meaningful
against income (see payment_burden).

Direct join from static_0.
'''

'''
credamount_770A
Requested credit amount.

Loan size sets both default odds (relative to
income) and loss-given-default.

Direct join from static_0.
'''
# Direct joins from static_0, preserving original missing values.
_feature_static = train_static_0_df.set_index("case_id")
df_master["days_employed_700P"] = df_master["case_id"].map(_feature_static["days_employed_700P"])
df_master["mainoccupationinc_384A"] = df_master["case_id"].map(_feature_static["mainoccupationinc_384A"])
df_master["annuity_780A"] = df_master["case_id"].map(_feature_static["annuity_780A"])
df_master["credamount_770A"] = df_master["case_id"].map(_feature_static["credamount_770A"])


In [40]:
'''
Family 2 · Bureau summary 2 features
The bureau's one-line read on the applicant (its risk score, recent query count) from static_cb_0. The highest-signal family — but
present only for applicants who already have a credit history, so it is empty for thin-file applicants.
'''

# Direct joins are repeated here to retain original missing values for the catalog features.
'''
riskassesment_940T
The bureau's own risk score (0–1).

Strongest single predictor — a pre-computed risk
opinion. Absent for thin-file applicants.

Direct join from static_cb_0; NaN
when no bureau record exists.
'''

'''
numberofqueries_146L
Count of recent credit-bureau checks.

Credit-seeking intensity; many queries signal stress
or credit hunger.

Direct join from static_cb_0.
'''
_feature_cb_summary = train_static_cb_0_df.set_index("case_id")
df_master["riskassesment_940T"] = df_master["case_id"].map(_feature_cb_summary["riskassesment_940T"])
df_master["numberofqueries_146L"] = df_master["case_id"].map(_feature_cb_summary["numberofqueries_146L"])


In [41]:
'''
Family 3 · Aggregates of depth-1 tables 40 features
The bulk of the set. Each table that has many rows per loan (bureau contracts, prior applications, tax, person records) is summarised
to one row with mean / std / min / max / sum / last. This captures the shape of a history — typical level, worst case, volatility, recency
— not just a single number.
'''

# Aggregate the raw depth-1 tables, not the previously imputed master table.
# "last" follows num_group1 order and skips missing values, as in the catalog.
# num_group1 is a record index; it does not prove chronological recency.
_feature_groups = {
    "credit_bureau_a_1": train_credit_bureau_a_1_df.sort_values(["case_id", "num_group1"]).groupby("case_id"),
    "person_1": train_person_1_df.sort_values(["case_id", "num_group1"]).groupby("case_id"),
    "applprev_1": train_applprev_1_df.sort_values(["case_id", "num_group1"]).groupby("case_id"),
    "tax_registry_a_1": train_tax_registry_a_1_df.sort_values(["case_id", "num_group1"]).groupby("case_id")
}
_feature_fields = {
    "credit_bureau_a_1": ["credamount_770A", "overdueamountmax_950A", "pmts_dpdvalue_108P"],
    "person_1": ["mainoccupationinc_384A", "empl_employedtotal_800L"],
    "applprev_1": ["credamount_590A"],
    "tax_registry_a_1": ["amount_4527230A"]
}
_feature_aggregates = {}
for _table, _fields in _feature_fields.items():
    _feature_aggregates[_table] = _feature_groups[_table][_fields].agg(
        ["mean", "std", "min", "max", "sum", "last"]
    )
    for _field in _fields:
        # All-missing amounts stay unknown instead of becoming a false zero.
        _feature_aggregates[_table][(_field, "sum")] = (
            _feature_groups[_table][_field].sum(min_count=1)
        )

'''
credit_bureau_a_1_pmts_dpdvalue_108P_mean
Mean of days past due across credit_bureau_a_1 records.

Summarises the typical level across available records.

groupby(case_id)["pmts_dpdvalue_108P"].mean() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_mean"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "mean")])
)

'''
credit_bureau_a_1_credamount_770A_mean
Mean of bureau credit amount across credit_bureau_a_1 records.

Summarises the typical level across available records.

groupby(case_id)["credamount_770A"].mean() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_mean"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "mean")])
)

'''
credit_bureau_a_1_pmts_dpdvalue_108P_max
Max of days past due across credit_bureau_a_1 records.

Captures the highest observed value.

groupby(case_id)["pmts_dpdvalue_108P"].max() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_max"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "max")])
)

'''
credit_bureau_a_1_pmts_dpdvalue_108P_sum
Sum of days past due across credit_bureau_a_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["pmts_dpdvalue_108P"].sum() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_sum"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "sum")])
)

'''
credit_bureau_a_1_pmts_dpdvalue_108P_last
Last of days past due across credit_bureau_a_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["pmts_dpdvalue_108P"].last() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_last"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "last")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_mean
Mean of maximum overdue amount across credit_bureau_a_1 records.

Summarises the typical level across available records.

groupby(case_id)["overdueamountmax_950A"].mean() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_mean"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "mean")])
)

'''
credit_bureau_a_1_credamount_770A_min
Min of bureau credit amount across credit_bureau_a_1 records.

Captures the lowest observed value.

groupby(case_id)["credamount_770A"].min() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_min"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "min")])
)

'''
credit_bureau_a_1_credamount_770A_max
Max of bureau credit amount across credit_bureau_a_1 records.

Captures the highest observed value.

groupby(case_id)["credamount_770A"].max() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_max"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "max")])
)

'''
tax_registry_a_1_amount_4527230A_mean
Mean of declared tax income across tax_registry_a_1 records.

Summarises the typical level across available records.

groupby(case_id)["amount_4527230A"].mean() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_mean"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "mean")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_max
Max of maximum overdue amount across credit_bureau_a_1 records.

Captures the highest observed value.

groupby(case_id)["overdueamountmax_950A"].max() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_max"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "max")])
)

'''
credit_bureau_a_1_credamount_770A_last
Last of bureau credit amount across credit_bureau_a_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["credamount_770A"].last() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_last"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "last")])
)

'''
tax_registry_a_1_amount_4527230A_max
Max of declared tax income across tax_registry_a_1 records.

Captures the highest observed value.

groupby(case_id)["amount_4527230A"].max() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_max"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "max")])
)

'''
tax_registry_a_1_amount_4527230A_min
Min of declared tax income across tax_registry_a_1 records.

Captures the lowest observed value.

groupby(case_id)["amount_4527230A"].min() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_min"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "min")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_sum
Sum of maximum overdue amount across credit_bureau_a_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["overdueamountmax_950A"].sum() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_sum"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "sum")])
)

'''
person_1_empl_employedtotal_800L_mean
Mean of person employment length across person_1 records.

Summarises the typical level across available records.

groupby(case_id)["empl_employedtotal_800L"].mean() on person_1.
'''
df_master["person_1_empl_employedtotal_800L_mean"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("empl_employedtotal_800L", "mean")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_last
Last of maximum overdue amount across credit_bureau_a_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["overdueamountmax_950A"].last() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_last"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "last")])
)

'''
person_1_mainoccupationinc_384A_mean
Mean of person income across person_1 records.

Summarises the typical level across available records.

groupby(case_id)["mainoccupationinc_384A"].mean() on person_1.
'''
df_master["person_1_mainoccupationinc_384A_mean"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("mainoccupationinc_384A", "mean")])
)

'''
tax_registry_a_1_amount_4527230A_last
Last of declared tax income across tax_registry_a_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["amount_4527230A"].last() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_last"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "last")])
)

'''
credit_bureau_a_1_pmts_dpdvalue_108P_std
Std of days past due across credit_bureau_a_1 records.

Measures variation across records; singleton histories remain missing.

groupby(case_id)["pmts_dpdvalue_108P"].std() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_std"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "std")])
)

'''
person_1_empl_employedtotal_800L_min
Min of person employment length across person_1 records.

Captures the lowest observed value.

groupby(case_id)["empl_employedtotal_800L"].min() on person_1.
'''
df_master["person_1_empl_employedtotal_800L_min"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("empl_employedtotal_800L", "min")])
)

'''
person_1_empl_employedtotal_800L_max
Max of person employment length across person_1 records.

Captures the highest observed value.

groupby(case_id)["empl_employedtotal_800L"].max() on person_1.
'''
df_master["person_1_empl_employedtotal_800L_max"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("empl_employedtotal_800L", "max")])
)

'''
credit_bureau_a_1_credamount_770A_std
Std of bureau credit amount across credit_bureau_a_1 records.

Measures variation across records; singleton histories remain missing.

groupby(case_id)["credamount_770A"].std() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_std"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "std")])
)

'''
credit_bureau_a_1_pmts_dpdvalue_108P_min
Min of days past due across credit_bureau_a_1 records.

Captures the lowest observed value.

groupby(case_id)["pmts_dpdvalue_108P"].min() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_pmts_dpdvalue_108P_min"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("pmts_dpdvalue_108P", "min")])
)

'''
person_1_empl_employedtotal_800L_last
Last of person employment length across person_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["empl_employedtotal_800L"].last() on person_1.
'''
df_master["person_1_empl_employedtotal_800L_last"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("empl_employedtotal_800L", "last")])
)

'''
person_1_mainoccupationinc_384A_max
Max of person income across person_1 records.

Captures the highest observed value.

groupby(case_id)["mainoccupationinc_384A"].max() on person_1.
'''
df_master["person_1_mainoccupationinc_384A_max"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("mainoccupationinc_384A", "max")])
)

'''
person_1_mainoccupationinc_384A_min
Min of person income across person_1 records.

Captures the lowest observed value.

groupby(case_id)["mainoccupationinc_384A"].min() on person_1.
'''
df_master["person_1_mainoccupationinc_384A_min"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("mainoccupationinc_384A", "min")])
)

'''
person_1_mainoccupationinc_384A_last
Last of person income across person_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["mainoccupationinc_384A"].last() on person_1.
'''
df_master["person_1_mainoccupationinc_384A_last"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("mainoccupationinc_384A", "last")])
)

'''
credit_bureau_a_1_credamount_770A_sum
Sum of bureau credit amount across credit_bureau_a_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["credamount_770A"].sum() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_credamount_770A_sum"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("credamount_770A", "sum")])
)

'''
applprev_1_credamount_590A_mean
Mean of prior requested credit across applprev_1 records.

Summarises the typical level across available records.

groupby(case_id)["credamount_590A"].mean() on applprev_1.
'''
df_master["applprev_1_credamount_590A_mean"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "mean")])
)

'''
tax_registry_a_1_amount_4527230A_sum
Sum of declared tax income across tax_registry_a_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["amount_4527230A"].sum() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_sum"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "sum")])
)

'''
person_1_empl_employedtotal_800L_sum
Sum of person employment length across person_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["empl_employedtotal_800L"].sum() on person_1.
'''
df_master["person_1_empl_employedtotal_800L_sum"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("empl_employedtotal_800L", "sum")])
)

'''
person_1_mainoccupationinc_384A_sum
Sum of person income across person_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["mainoccupationinc_384A"].sum() on person_1.
'''
df_master["person_1_mainoccupationinc_384A_sum"] = (
    df_master["case_id"].map(_feature_aggregates["person_1"][("mainoccupationinc_384A", "sum")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_std
Std of maximum overdue amount across credit_bureau_a_1 records.

Measures variation across records; singleton histories remain missing.

groupby(case_id)["overdueamountmax_950A"].std() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_std"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "std")])
)

'''
applprev_1_credamount_590A_max
Max of prior requested credit across applprev_1 records.

Captures the highest observed value.

groupby(case_id)["credamount_590A"].max() on applprev_1.
'''
df_master["applprev_1_credamount_590A_max"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "max")])
)

'''
applprev_1_credamount_590A_min
Min of prior requested credit across applprev_1 records.

Captures the lowest observed value.

groupby(case_id)["credamount_590A"].min() on applprev_1.
'''
df_master["applprev_1_credamount_590A_min"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "min")])
)

'''
applprev_1_credamount_590A_last
Last of prior requested credit across applprev_1 records.

Captures the last nonmissing value in record-index order.

groupby(case_id)["credamount_590A"].last() on applprev_1.
'''
df_master["applprev_1_credamount_590A_last"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "last")])
)

'''
applprev_1_credamount_590A_sum
Sum of prior requested credit across applprev_1 records.

Captures the total across observed records; all-missing groups remain missing.

groupby(case_id)["credamount_590A"].sum() on applprev_1.
'''
df_master["applprev_1_credamount_590A_sum"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "sum")])
)

'''
credit_bureau_a_1_overdueamountmax_950A_min
Min of maximum overdue amount across credit_bureau_a_1 records.

Captures the lowest observed value.

groupby(case_id)["overdueamountmax_950A"].min() on credit_bureau_a_1.
'''
df_master["credit_bureau_a_1_overdueamountmax_950A_min"] = (
    df_master["case_id"].map(_feature_aggregates["credit_bureau_a_1"][("overdueamountmax_950A", "min")])
)

'''
tax_registry_a_1_amount_4527230A_std
Std of declared tax income across tax_registry_a_1 records.

Measures variation across records; singleton histories remain missing.

groupby(case_id)["amount_4527230A"].std() on tax_registry_a_1.
'''
df_master["tax_registry_a_1_amount_4527230A_std"] = (
    df_master["case_id"].map(_feature_aggregates["tax_registry_a_1"][("amount_4527230A", "std")])
)

'''
applprev_1_credamount_590A_std
Std of prior requested credit across applprev_1 records.

Measures variation across records; singleton histories remain missing.

groupby(case_id)["credamount_590A"].std() on applprev_1.
'''
df_master["applprev_1_credamount_590A_std"] = (
    df_master["case_id"].map(_feature_aggregates["applprev_1"][("credamount_590A", "std")])
)

_catalog_aggregate_features = ['credit_bureau_a_1_pmts_dpdvalue_108P_mean', 'credit_bureau_a_1_credamount_770A_mean', 'credit_bureau_a_1_pmts_dpdvalue_108P_max', 'credit_bureau_a_1_pmts_dpdvalue_108P_sum', 'credit_bureau_a_1_pmts_dpdvalue_108P_last', 'credit_bureau_a_1_overdueamountmax_950A_mean', 'credit_bureau_a_1_credamount_770A_min', 'credit_bureau_a_1_credamount_770A_max', 'tax_registry_a_1_amount_4527230A_mean', 'credit_bureau_a_1_overdueamountmax_950A_max', 'credit_bureau_a_1_credamount_770A_last', 'tax_registry_a_1_amount_4527230A_max', 'tax_registry_a_1_amount_4527230A_min', 'credit_bureau_a_1_overdueamountmax_950A_sum', 'person_1_empl_employedtotal_800L_mean', 'credit_bureau_a_1_overdueamountmax_950A_last', 'person_1_mainoccupationinc_384A_mean', 'tax_registry_a_1_amount_4527230A_last', 'credit_bureau_a_1_pmts_dpdvalue_108P_std', 'person_1_empl_employedtotal_800L_min', 'person_1_empl_employedtotal_800L_max', 'credit_bureau_a_1_credamount_770A_std', 'credit_bureau_a_1_pmts_dpdvalue_108P_min', 'person_1_empl_employedtotal_800L_last', 'person_1_mainoccupationinc_384A_max', 'person_1_mainoccupationinc_384A_min', 'person_1_mainoccupationinc_384A_last', 'credit_bureau_a_1_credamount_770A_sum', 'applprev_1_credamount_590A_mean', 'tax_registry_a_1_amount_4527230A_sum', 'person_1_empl_employedtotal_800L_sum', 'person_1_mainoccupationinc_384A_sum', 'credit_bureau_a_1_overdueamountmax_950A_std', 'applprev_1_credamount_590A_max', 'applprev_1_credamount_590A_min', 'applprev_1_credamount_590A_last', 'applprev_1_credamount_590A_sum', 'credit_bureau_a_1_overdueamountmax_950A_min', 'tax_registry_a_1_amount_4527230A_std', 'applprev_1_credamount_590A_std']


In [42]:
'''
Family 4 · Coverage & presence flags 7 features
Whether each data source exists for this applicant (thin_file, has_bureau, record counts). Absence is itself signal, and this
family defines the thin-file segment the project is about.
'''

# Row counts include records even when their numeric values are missing.
_feature_bureau_count = df_master["case_id"].map(_feature_groups["credit_bureau_a_1"].size()).fillna(0)
_feature_prev_count = df_master["case_id"].map(_feature_groups["applprev_1"].size()).fillna(0)
_feature_tax_count = df_master["case_id"].map(_feature_groups["tax_registry_a_1"].size()).fillna(0)

'''
has_bureau
1 if any bureau contract exists.

Presence of a credit history.

credit_bureau_a_1 row count > 0.
'''
df_master["has_bureau"] = (
    (_feature_bureau_count > 0).astype(int)
)

'''
has_cb_summary
1 if a bureau summary row exists.

Independent of contract presence.

base.case_id in static_cb_0.case_id.
'''
df_master["has_cb_summary"] = (
    df_master["case_id"].isin(train_static_cb_0_df["case_id"]).astype(int)
)

'''
thin_file
1 if no bureau contracts exist.

Defines the thin-file segment.

credit_bureau_a_1 row count == 0.
'''
df_master["thin_file"] = (
    (_feature_bureau_count == 0).astype(int)
)

'''
has_prev
1 if prior applications exist.

Captures a prior lender relationship.

applprev_1 row count > 0.
'''
df_master["has_prev"] = (
    (_feature_prev_count > 0).astype(int)
)

'''
has_coapplicant
1 if more than one person is on the application.

Distinguishes joint and solo applications.

person_1 row count > 1.
'''
df_master["has_coapplicant"] = (
    (df_master["case_id"].map(_feature_groups["person_1"].size()).fillna(0) > 1).astype(int)
)

'''
has_tax
1 if tax records exist.

Captures availability of tax income records.

tax_registry_a_1 row count > 0.
'''
df_master["has_tax"] = (
    (_feature_tax_count > 0).astype(int)
)

'''
person_count
Number of people on the application.

Captures household application size.

groupby(case_id).size() on person_1.
'''
df_master["person_count"] = (
    df_master["case_id"].map(_feature_groups["person_1"].size()).fillna(0).astype(int)
)



In [43]:
'''
Family 5 · Affordability & leverage ratios 7 features
Raw amounts made meaningful by dividing by income — payment burden, leverage, existing exposure. The “can they actually afford
it” view, and crucially it works for everyone, including thin-file applicants.
'''

# Missing, zero or negative denominators produce NaN, never infinity.
_feature_income = df_master["mainoccupationinc_384A"].where(df_master["mainoccupationinc_384A"] > 0)
_feature_credit = df_master["credamount_770A"].where(df_master["credamount_770A"] > 0)
_feature_annuity = df_master["annuity_780A"].where(df_master["annuity_780A"] > 0)
_feature_bureau_sum = df_master["credit_bureau_a_1_credamount_770A_sum"].where(
    df_master["credit_bureau_a_1_credamount_770A_sum"] > 0
)

'''
bureau_exposure_ratio
Total past credit divided by income.

Existing credit exposure relative to repayment capacity.

credit_bureau_a_1_credamount_770A_sum / mainoccupationinc_384A.
'''
df_master["bureau_exposure_ratio"] = (
    df_master["credit_bureau_a_1_credamount_770A_sum"] / _feature_income
)

'''
leverage
Requested credit divided by income.

Loan size relative to capacity.

credamount_770A / mainoccupationinc_384A.
'''
df_master["leverage"] = (
    df_master["credamount_770A"] / _feature_income
)

'''
payment_burden
Annuity divided by income.

Payment obligation relative to capacity.

annuity_780A / mainoccupationinc_384A.
'''
df_master["payment_burden"] = (
    df_master["annuity_780A"] / _feature_income
)

'''
prior_credit_to_current
Prior mean requested credit divided by current requested credit.

Change in borrowing appetite.

applprev_1_credamount_590A_mean / credamount_770A.
'''
df_master["prior_credit_to_current"] = (
    df_master["applprev_1_credamount_590A_mean"] / _feature_credit
)

'''
income_consistency
Declared tax income divided by stated income.

Captures income-source mismatch.

tax_registry_a_1_amount_4527230A_mean / mainoccupationinc_384A.
'''
df_master["income_consistency"] = (
    df_master["tax_registry_a_1_amount_4527230A_mean"] / _feature_income
)

'''
implied_term
Credit divided by annuity.

Rough payment-count proxy, without adjusting for interest.

credamount_770A / annuity_780A.
'''
df_master["implied_term"] = (
    df_master["credamount_770A"] / _feature_annuity
)

'''
credit_concentration
Largest past credit line divided by total past credit.

Concentration of credit exposure.

credit_bureau_a_1_credamount_770A_max / credit_bureau_a_1_credamount_770A_sum.
'''
df_master["credit_concentration"] = (
    df_master["credit_bureau_a_1_credamount_770A_max"] / _feature_bureau_sum
)



In [44]:
'''
Family 6 · Credit-behaviour ratios 6 features
How the applicant handled past credit, normalised for history length — share of contracts overdue, share ever late, severity per
contract. Behaviour, not just balances.
'''

_feature_contract_denominator = _feature_bureau_count.where(_feature_bureau_count > 0)
_feature_month_count = df_master["case_id"].map(
    train_credit_bureau_a_2_df.groupby("case_id").size()
).fillna(0)

'''
dpd_contract_rate
Share of bureau contracts with positive days past due.

Normalises delinquency frequency for history length.

Per-case mean of 1[pmts_dpdvalue_108P > 0].
'''
df_master["dpd_contract_rate"] = (
    df_master["case_id"].map(
        train_credit_bureau_a_1_df.assign(
            _has_dpd=lambda x: (x["pmts_dpdvalue_108P"] > 0).astype(int)
        ).groupby("case_id")["_has_dpd"].mean()
    )
)

'''
avg_overdue_per_contract
Overdue amount per bureau contract.

Delinquency severity relative to history size.

credit_bureau_a_1_overdueamountmax_950A_sum / contract count.
'''
df_master["avg_overdue_per_contract"] = (
    df_master["credit_bureau_a_1_overdueamountmax_950A_sum"] / _feature_contract_denominator
)

'''
queries_per_contract
Bureau queries per contract, with an offset of one.

Credit-seeking intensity relative to existing history.

numberofqueries_146L / (contract count + 1).
'''
df_master["queries_per_contract"] = (
    df_master["numberofqueries_146L"] / (_feature_bureau_count + 1)
)

'''
max_overdue_to_income
Worst overdue amount divided by income.

Worst delinquency relative to repayment capacity.

credit_bureau_a_1_overdueamountmax_950A_max / mainoccupationinc_384A.
'''
df_master["max_overdue_to_income"] = (
    df_master["credit_bureau_a_1_overdueamountmax_950A_max"] / _feature_income
)

# The PDF defines a mean of row indicators: missing overdue compares False.
# missing_overdue below separately flags cases without any observed overdue value.
'''
overdue_contract_rate
Share of bureau rows with a positive observed overdue amount.

Normalises recorded delinquency frequency for history length.

Per-case mean of 1[overdueamountmax_950A > 0] over all bureau rows.
'''
df_master["overdue_contract_rate"] = (
    df_master["case_id"].map(
        train_credit_bureau_a_1_df.assign(
            _has_overdue=lambda x: (x["overdueamountmax_950A"] > 0).astype(int)
        ).groupby("case_id")["_has_overdue"].mean()
    )
)

'''
months_per_contract
Payment-history rows per bureau contract.

Depth of recorded payment history; rows need not be distinct months.

credit_bureau_a_2 row count / credit_bureau_a_1 row count.
'''
df_master["months_per_contract"] = (
    _feature_month_count / _feature_contract_denominator
)



In [45]:
'''
Family 7 · Demographic / derived 3 features
Lifecycle and household context — age, income relative to tenure, how much a co-applicant adds.
'''

'''
income_per_employed_year
Income normalised by years employed, with an offset of one.

Relates earnings to employment tenure.

mainoccupationinc_384A / (days_employed_700P / 365 + 1).
'''
df_master["income_per_employed_year"] = (
    df_master["mainoccupationinc_384A"] / (
        df_master["days_employed_700P"] / 365 + 1
    ).where((df_master["days_employed_700P"] / 365 + 1) > 0)
)

'''
age
Applicant age in years at the decision date.

Lifecycle context; included because the catalog lists it.

(date_decision - birth_259D).days / 365.25.
'''
df_master["age"] = (
    (
        pd.to_datetime(df_master["date_decision"], errors="coerce")
        - pd.to_datetime(df_master["case_id"].map(_feature_static["birth_259D"]), errors="coerce")
    ).dt.days / 365.25
)

'''
coapplicant_income_ratio
Lowest person income divided by highest person income.

Measures income balance; a solo observed person gives 1.

person_1_mainoccupationinc_384A_min / person_1_mainoccupationinc_384A_max.
'''
df_master["coapplicant_income_ratio"] = (
    df_master["person_1_mainoccupationinc_384A_min"] / df_master[
        "person_1_mainoccupationinc_384A_max"
    ].where(df_master["person_1_mainoccupationinc_384A_max"] > 0)
)



In [46]:
'''
Family 8 · Categorical encodings 19 features
The catalog lists education, bureau contract class and bureau description, one-hot encoded so a tree can split on a specific category.
Protected, identifier and noise categoricals were removed by the screen.
'''

# Consolidate columns before adding the categorical indicators.
df_master = df_master.copy()

# The detailed list includes description_5085714M although the overview mentions two fields.
# Choose the per-case contract-class mode; ties resolve lexicographically.
_feature_class_counts = (
    train_credit_bureau_a_1_df.dropna(subset=["classificationofcontr_13M"])
    .groupby(["case_id", "classificationofcontr_13M"]).size().reset_index(name="_count")
)
_feature_class_mode = (
    _feature_class_counts.sort_values(
        ["case_id", "_count", "classificationofcontr_13M"], ascending=[True, False, True]
    ).drop_duplicates("case_id").set_index("case_id")["classificationofcontr_13M"]
)
_feature_categories = {
    "education_927M": df_master["case_id"].map(_feature_static["education_927M"]),
    "classificationofcontr_13M": df_master["case_id"].map(_feature_class_mode),
    "description_5085714M": df_master["case_id"].map(_feature_cb_summary["description_5085714M"])
}
# Use the explicit catalog categories, matching the existing manual one-hot format.
# No rare-pooling threshold is specified by the PDF; no arbitrary pooling is added.
# The listed "nan" columns explicitly encode missing class/description.

'''
oh_education_927M_2d4a2bc4
One-hot indicator (1/0) for education_927M category 2d4a2bc4.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case education_927M == 2d4a2bc4]; contract class uses the per-case mode.
'''
df_master["oh_education_927M_2d4a2bc4"] = (
    (_feature_categories["education_927M"] == "2d4a2bc4").astype(int)
)

'''
oh_classificationofcontr_13M_54ddc605
One-hot indicator (1/0) for classificationofcontr_13M category 54ddc605.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == 54ddc605]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_54ddc605"] = (
    (_feature_categories["classificationofcontr_13M"] == "54ddc605").astype(int)
)

'''
oh_description_5085714M_34bbcd2a
One-hot indicator (1/0) for description_5085714M category 34bbcd2a.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 34bbcd2a]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_34bbcd2a"] = (
    (_feature_categories["description_5085714M"] == "34bbcd2a").astype(int)
)

'''
oh_education_927M_10795bac
One-hot indicator (1/0) for education_927M category 10795bac.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case education_927M == 10795bac]; contract class uses the per-case mode.
'''
df_master["oh_education_927M_10795bac"] = (
    (_feature_categories["education_927M"] == "10795bac").astype(int)
)

'''
oh_education_927M_f937ecaa
One-hot indicator (1/0) for education_927M category f937ecaa.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case education_927M == f937ecaa]; contract class uses the per-case mode.
'''
df_master["oh_education_927M_f937ecaa"] = (
    (_feature_categories["education_927M"] == "f937ecaa").astype(int)
)

'''
oh_classificationofcontr_13M_6f5666c7
One-hot indicator (1/0) for classificationofcontr_13M category 6f5666c7.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == 6f5666c7]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_6f5666c7"] = (
    (_feature_categories["classificationofcontr_13M"] == "6f5666c7").astype(int)
)

'''
oh_description_5085714M_nan
One-hot indicator (1/0) for description_5085714M category nan.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == nan]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_nan"] = (
    _feature_categories["description_5085714M"].isna().astype(int)
)

'''
oh_classificationofcontr_13M_nan
One-hot indicator (1/0) for classificationofcontr_13M category nan.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == nan]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_nan"] = (
    _feature_categories["classificationofcontr_13M"].isna().astype(int)
)

'''
oh_description_5085714M_5568a5e6
One-hot indicator (1/0) for description_5085714M category 5568a5e6.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 5568a5e6]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_5568a5e6"] = (
    (_feature_categories["description_5085714M"] == "5568a5e6").astype(int)
)

'''
oh_education_927M_242b264b
One-hot indicator (1/0) for education_927M category 242b264b.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case education_927M == 242b264b]; contract class uses the per-case mode.
'''
df_master["oh_education_927M_242b264b"] = (
    (_feature_categories["education_927M"] == "242b264b").astype(int)
)

'''
oh_description_5085714M_6ef3de40
One-hot indicator (1/0) for description_5085714M category 6ef3de40.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 6ef3de40]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_6ef3de40"] = (
    (_feature_categories["description_5085714M"] == "6ef3de40").astype(int)
)

'''
oh_classificationofcontr_13M_d8bf6dd7
One-hot indicator (1/0) for classificationofcontr_13M category d8bf6dd7.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == d8bf6dd7]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_d8bf6dd7"] = (
    (_feature_categories["classificationofcontr_13M"] == "d8bf6dd7").astype(int)
)

'''
oh_description_5085714M_3802c3d2
One-hot indicator (1/0) for description_5085714M category 3802c3d2.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 3802c3d2]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_3802c3d2"] = (
    (_feature_categories["description_5085714M"] == "3802c3d2").astype(int)
)

'''
oh_classificationofcontr_13M_fe64f125
One-hot indicator (1/0) for classificationofcontr_13M category fe64f125.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == fe64f125]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_fe64f125"] = (
    (_feature_categories["classificationofcontr_13M"] == "fe64f125").astype(int)
)

'''
oh_education_927M_6def22f0
One-hot indicator (1/0) for education_927M category 6def22f0.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case education_927M == 6def22f0]; contract class uses the per-case mode.
'''
df_master["oh_education_927M_6def22f0"] = (
    (_feature_categories["education_927M"] == "6def22f0").astype(int)
)

'''
oh_classificationofcontr_13M_01851564
One-hot indicator (1/0) for classificationofcontr_13M category 01851564.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == 01851564]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_01851564"] = (
    (_feature_categories["classificationofcontr_13M"] == "01851564").astype(int)
)

'''
oh_classificationofcontr_13M_ba38dde4
One-hot indicator (1/0) for classificationofcontr_13M category ba38dde4.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case classificationofcontr_13M == ba38dde4]; contract class uses the per-case mode.
'''
df_master["oh_classificationofcontr_13M_ba38dde4"] = (
    (_feature_categories["classificationofcontr_13M"] == "ba38dde4").astype(int)
)

'''
oh_description_5085714M_4dc79565
One-hot indicator (1/0) for description_5085714M category 4dc79565.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 4dc79565]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_4dc79565"] = (
    (_feature_categories["description_5085714M"] == "4dc79565").astype(int)
)

'''
oh_description_5085714M_521190c5
One-hot indicator (1/0) for description_5085714M category 521190c5.

Allows a model to distinguish this category without imposing ordinal order.

1[per-case description_5085714M == 521190c5]; contract class uses the per-case mode.
'''
df_master["oh_description_5085714M_521190c5"] = (
    (_feature_categories["description_5085714M"] == "521190c5").astype(int)
)

_catalog_encoding_features = ['oh_education_927M_2d4a2bc4', 'oh_classificationofcontr_13M_54ddc605', 'oh_description_5085714M_34bbcd2a', 'oh_education_927M_10795bac', 'oh_education_927M_f937ecaa', 'oh_classificationofcontr_13M_6f5666c7', 'oh_description_5085714M_nan', 'oh_classificationofcontr_13M_nan', 'oh_description_5085714M_5568a5e6', 'oh_education_927M_242b264b', 'oh_description_5085714M_6ef3de40', 'oh_classificationofcontr_13M_d8bf6dd7', 'oh_description_5085714M_3802c3d2', 'oh_classificationofcontr_13M_fe64f125', 'oh_education_927M_6def22f0', 'oh_classificationofcontr_13M_01851564', 'oh_classificationofcontr_13M_ba38dde4', 'oh_description_5085714M_4dc79565', 'oh_description_5085714M_521190c5']


In [47]:
'''
Family 9 · Missing-data indicators 4 features
Flags for whether key fields were present. Here missingness is structural (a field is absent because the underlying record does not
exist), so the flag itself is informative — see the dedicated page.

Status: Done
'''

'''
missing_overdue
1 if no observed overdue data.

Preserves source missingness separately from the numeric value.

credit_bureau_a_1_overdueamountmax_950A_mean.isna() before model imputation.
'''
df_master["missing_overdue"] = (
    df_master["credit_bureau_a_1_overdueamountmax_950A_mean"].isna().astype(int)
)

'''
missing_annuity
1 if the source annuity was missing.

Preserves source missingness separately from the numeric value.

annuity_780A.isna() before model imputation.
'''
df_master["missing_annuity"] = (
    df_master["annuity_780A"].isna().astype(int)
)

'''
missing_days_employed
1 if source employment length was missing.

Preserves source missingness separately from the numeric value.

days_employed_700P.isna() before model imputation.
'''
df_master["missing_days_employed"] = (
    df_master["days_employed_700P"].isna().astype(int)
)

'''
missing_income
1 if source stated income was missing.

Preserves source missingness separately from the numeric value.

mainoccupationinc_384A.isna() before model imputation.
'''
df_master["missing_income"] = (
    df_master["mainoccupationinc_384A"].isna().astype(int)
)



In [48]:
# Check all nine catalog families: 4 + 2 + 40 + 7 + 7 + 6 + 3 + 19 + 4 = 92.
_catalog_features = (
    ["days_employed_700P", "mainoccupationinc_384A", "annuity_780A", "credamount_770A"]
    + ["riskassesment_940T", "numberofqueries_146L"]
    + _catalog_aggregate_features
    + ["has_bureau", "has_cb_summary", "thin_file", "has_prev", "has_coapplicant", "has_tax", "person_count"]
    + ["bureau_exposure_ratio", "leverage", "payment_burden", "prior_credit_to_current", "income_consistency", "implied_term", "credit_concentration"]
    + ["dpd_contract_rate", "avg_overdue_per_contract", "queries_per_contract", "max_overdue_to_income", "overdue_contract_rate", "months_per_contract"]
    + ["income_per_employed_year", "age", "coapplicant_income_ratio"]
    + _catalog_encoding_features
    + ["missing_overdue", "missing_annuity", "missing_days_employed", "missing_income"]
)
assert len(_catalog_features) == len(set(_catalog_features)) == 92
assert set(_catalog_features).issubset(df_master.columns)
assert df_master["case_id"].is_unique
assert not np.isinf(df_master[_catalog_features].to_numpy(dtype=float)).any()
assert (df_master["has_bureau"] + df_master["thin_file"]).eq(1).all()
assert df_master["credit_concentration"].dropna().between(0, 1).all()

df_master.head()


,case_id,date_decision,MONTH,WEEK_NUM,mainoccupationinc_384A,credamount_770A,annuity_780A,days_employed_700P,education_927M,maritalstatus_703M,...,oh_classificationofcontr_13M_fe64f125,oh_education_927M_6def22f0,oh_classificationofcontr_13M_01851564,oh_classificationofcontr_13M_ba38dde4,oh_description_5085714M_4dc79565,oh_description_5085714M_521190c5,missing_overdue,missing_annuity,missing_days_employed,missing_income
0,1000001,2020-11-03,202011,96,28384.0,7397.0,1188.0,661.0,2d4a2bc4,ea6fb4b5,...,0,0,0,0,0,0,1,0,0,0
1,1000002,2020-10-27,202010,95,40770.0,16124.0,637.0,2766.0,f937ecaa,b58630a7,...,1,0,0,0,0,0,0,0,0,0
2,1000003,2020-10-06,202010,92,25052.0,30771.0,749.0,0.0,6def22f0,b58630a7,...,0,1,1,0,1,0,1,0,0,0
3,1000004,2020-12-15,202012,102,16929.0,11339.0,1467.0,1768.0,242b264b,ea6fb4b5,...,0,0,0,0,0,0,0,0,0,0
4,1000005,2020-11-17,202011,98,20529.0,15124.0,1760.0,2067.0,f937ecaa,e78cb9c0,...,0,0,0,0,0,0,1,0,0,0


In [49]:
id_cols = [c for c in ["case_id", "WEEK_NUM", "target"] if c in df_master.columns]
df_model = df_master[id_cols + _catalog_features].copy()
df_model.to_parquet(f"../data/{SPLIT}_features.parquet")
print(df_model.shape)

(200000, 94)


In [50]:
print((df_master["days_employed_700P"] == 365243).sum())
print(df_master["days_employed_700P"].describe())
print(df_master["age"].describe())

0
count    169970.000000
mean       1522.220492
std         901.477904
min           0.000000
25%         861.000000
50%        1497.000000
75%        2142.000000
max        5923.000000
Name: days_employed_700P, dtype: float64
count    200000.000000
mean         47.343927
std          14.118025
min          22.748802
25%          35.099247
50%          47.340178
75%          59.561944
max          71.994524
Name: age, dtype: float64
